In [1]:
%pip -q install plotly ipywidgets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 38.4 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import plotly.express as px
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
from google.colab import files

In [5]:
uploaded = files.upload()
nama_file = next(iter(uploaded))
data = pd.read_csv(nama_file)
print("Ukuran data:", data.shape)
display(data.head())

Saving dataset_dummy_penjualan.csv to dataset_dummy_penjualan (1).csv
Ukuran data: (360, 8)


,tanggal,produk,kategori,wilayah,kanal_penjualan,jumlah_terjual,harga_satuan_rp,pendapatan_rp
0,2025-11-24,Laptop,Elektronik,Jakarta,Toko Online,1,7862325,7862325
1,2025-03-13,Webcam,Aksesori,Yogyakarta,Toko Online,2,621203,1242406
2,2025-10-30,Monitor,Elektronik,Jakarta,Toko Online,1,1993110,1993110
3,2025-09-16,Headset,Aksesori,Yogyakarta,Toko Fisik,1,455512,455512
4,2025-04-23,Monitor,Elektronik,Bengkulu,Toko Online,5,2336149,11680745


In [6]:
data['tanggal'] = pd.to_datetime(data['tanggal'], errors='coerce')
data['wilayah'] = data['wilayah'].fillna('Tidak diketahui')
data = data.drop_duplicates().copy()
data = data.dropna(subset=['tanggal', 'produk', 'pendapatan_rp'])
data['pendapatan_rp'] = pd.to_numeric(data['pendapatan_rp'], errors='coerce')
data['jumlah_terjual'] = pd.to_numeric(data['jumlah_terjual'], errors='coerce')
data = data.dropna(subset=['pendapatan_rp', 'jumlah_terjual'])
data['bulan'] = data['tanggal'].dt.to_period('M').astype(str)
print("Jumlah baris setelah pembersihan:", len(data))
display(data.isna().sum().to_frame('jumlah_nilai_kosong'))

Jumlah baris setelah pembersihan: 360


,jumlah_nilai_kosong
tanggal,0
produk,0
kategori,0
wilayah,0
kanal_penjualan,0
jumlah_terjual,0
harga_satuan_rp,0
pendapatan_rp,0
bulan,0


In [7]:
wilayah_opsi = ['Semua'] + sorted(data['wilayah'].dropna().unique().tolist())
kategori_opsi = ['Semua'] + sorted(data['kategori'].dropna().unique().tolist())
pilih_wilayah = widgets.Dropdown(options=wilayah_opsi, value='Semua', description='Wilayah:')
pilih_kategori = widgets.Dropdown(options=kategori_opsi, value='Semua', description='Kategori:')
output = widgets.Output()

def tampilkan_dashboard(_=None):
    with output:
        clear_output(wait=True)
        hasil = data.copy()
        if pilih_wilayah.value != 'Semua':
            hasil = hasil[hasil['wilayah'] == pilih_wilayah.value]
        if pilih_kategori.value != 'Semua':
            hasil = hasil[hasil['kategori'] == pilih_kategori.value]
        if hasil.empty:
            display(Markdown('**Tidak ada data** untuk filter ini. Silakan ubah pilihan.'))
            return

        display(Markdown('## Ringkasan (Overview First)'))
        ringkasan = pd.DataFrame({
            'Indikator': ['Total pendapatan', 'Unit terjual', 'Jumlah catatan', 'Rata-rata pendapatan per catatan'],
            'Nilai': [
                f"Rp {hasil['pendapatan_rp'].sum():,.0f}",
                f"{hasil['jumlah_terjual'].sum():,.0f}",
                f"{len(hasil):,.0f}",
                f"Rp {hasil['pendapatan_rp'].mean():,.0f}"
            ]
        })
        display(ringkasan)

        tren = hasil.groupby('bulan', as_index=False)['pendapatan_rp'].sum()
        fig = px.line(tren, x='bulan', y='pendapatan_rp', title='Tren Pendapatan per Bulan',
                      labels={'bulan':'Bulan','pendapatan_rp':'Pendapatan (Rp)'})
        fig.update_layout(xaxis_tickangle=-45)
        fig.show()

        kategori = hasil.groupby('kategori', as_index=False)['pendapatan_rp'].sum().sort_values('pendapatan_rp', ascending=False)
        px.bar(kategori, x='kategori', y='pendapatan_rp', title='Pendapatan per Kategori',
               labels={'kategori':'Kategori','pendapatan_rp':'Pendapatan (Rp)'}).show()

        wilayah = hasil.groupby('wilayah', as_index=False)['pendapatan_rp'].sum().sort_values('pendapatan_rp', ascending=False)
        px.bar(wilayah, x='wilayah', y='pendapatan_rp', title='Pendapatan per Wilayah',
               labels={'wilayah':'Wilayah','pendapatan_rp':'Pendapatan (Rp)'}).show()

        display(Markdown('## Detail-on-Demand: 10 catatan terbaru'))
        display(hasil.sort_values('tanggal', ascending=False)[
            ['tanggal','produk','kategori','wilayah','kanal_penjualan','jumlah_terjual','pendapatan_rp']
        ].head(10))

pilih_wilayah.observe(tampilkan_dashboard, names='value')
pilih_kategori.observe(tampilkan_dashboard, names='value')
display(widgets.HBox([pilih_wilayah, pilih_kategori]))
display(output)
tampilkan_dashboard()

Output()

In [8]:
nama_output = 'data_penjualan_bersih.csv'
data.to_csv(nama_output, index=False)
print('File siap diunduh:', nama_output)
files.download(nama_output)

File siap diunduh: data_penjualan_bersih.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>